In [ ]:
import pandas as pd
import numpy as np
import os, requests
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
from io import BytesIO

After load:            8983
After dropna:          8575
After outlier removal: 8406
Downloaded: 8406 / 8406
Saved listings_clean.csv


In [ ]:
# paths
BASE_DIR  = os.getcwd()
IMAGE_DIR = os.path.join(BASE_DIR, 'images')
os.makedirs(IMAGE_DIR, exist_ok=True)

In [ ]:
# load
df = pd.read_csv(os.path.join(BASE_DIR, 'housing.csv'))
print(f"After load:            {len(df)}")

In [ ]:
# clean csv
df = df.drop(columns=['Unnamed: 2', 'Unnamed: 40'], errors='ignore')

df['price'] = pd.to_numeric(df['price'].astype(str).str.replace(r'[\$,\s]', '', regex=True), errors='coerce')

for col in ['sqft', 'sqft_lot']:
    if df[col].dtype == object:
        df[col] = df[col].str.replace(',', '').str.extract(r'(\d+\.?\d*)').astype(float)

if df['price_per_sqft'].dtype == object:
    df['price_per_sqft'] = df['price_per_sqft'].str.replace(r'[\$,/sqft\s]', '', regex=True).astype(float)

for col in ['walk_score', 'bike_score']:
    if col in df.columns and df[col].dtype == object:
        df[col] = df[col].str.extract(r'(\d+)').astype(float)

if df['estimated_monthly_payment'].dtype == object:
    df['estimated_monthly_payment'] = df['estimated_monthly_payment'].str.extract(r'(\d+\.?\d*)').astype(float)

df['property_type'] = df['property_type'].str.lower().str.strip()
df = df.drop(columns=['transit_score'], errors='ignore')

for col in ['baths', 'year_built', 'walk_score', 'bike_score', 'parking_total_spaces']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].median())

df = df.dropna(subset=['price', 'image_url', 'beds', 'sqft'])
print(f"After dropna:          {len(df)}")

q_low  = df['price'].quantile(0.01)
q_high = df['price'].quantile(0.99)
df = df[(df['price'] >= q_low) & (df['price'] <= q_high)]
print(f"After outlier removal: {len(df)}")

In [ ]:
# download images
def download_image(row):
    img_path = os.path.join(IMAGE_DIR, f"{row.name}.jpg")
    if os.path.exists(img_path):
        return img_path
    try:
        r = requests.get(row['image_url'], timeout=10, headers={'User-Agent': 'Mozilla/5.0'})
        img = Image.open(BytesIO(r.content)).convert('RGB')
        img = img.resize((224, 224))
        img.save(img_path)
        return img_path
    except:
        return None

print("Downloading images...")
with ThreadPoolExecutor(max_workers=8) as ex:
    df['img_path'] = list(ex.map(download_image, [row for _, row in df.iterrows()]))

print(f"Downloaded: {df['img_path'].notna().sum()} / {len(df)}")

In [ ]:
# save cleaned csv
df.to_csv(os.path.join(BASE_DIR, 'listings_clean.csv'), index=False)
print("Saved listings_clean.csv")